# ARC Prize 2026 — ARC-AGI-3 Submission

Built from `agent/my_agent.py` via `scripts/build_notebook.py`. Do not edit cells directly — edit the source file and re-run `make submit`.

In [ ]:
!pip install --no-index --find-links \
    /kaggle/input/competitions/arc-prize-2026-arc-agi-3/arc_agi_3_wheels \
    arc-agi python-dotenv

In [ ]:
%%writefile /tmp/my_agent.py
"""Adaptive, deterministic baseline agent for ARC-AGI-3.

Design goals:
- legal-action aware: never emit an action outside the current action space;
- information efficient: probe simple controls before spending coordinate actions;
- reproducible: stable per-game RNG and bounded action budget;
- competition safe: no filesystem/network/model dependencies at decision time.

This is a baseline, not a claim of human-level generalization. The intended
next step is to add game-family-specific policies behind the same interface.
"""
from __future__ import annotations

import hashlib
import random
from typing import Any

from arcengine import FrameData, GameAction, GameState
from agents.agent import Agent


class MyAgent(Agent):
    """Budgeted adaptive exploration with deterministic replay behavior."""

    MAX_ACTIONS = 240
    PROBE_PERIOD = 6

    def __init__(self, *args: Any, **kwargs: Any) -> None:
        super().__init__(*args, **kwargs)
        digest = hashlib.sha256(self.game_id.encode("utf-8")).digest()
        self._rng = random.Random(int.from_bytes(digest[:8], "big"))
        self._turn = 0
        self._seen_action_names: set[str] = set()
        self._frame_signatures: set[str] = set()
        self._stagnant_steps = 0
        self._last_signature = ""

    @property
    def name(self) -> str:
        return f"{super().name}.adaptive240"

    @staticmethod
    def _signature(frame: FrameData) -> str:
        """Use stable public frame fields without assuming a game-specific schema."""
        parts: list[str] = []
        for attr in ("state", "levels_completed", "available_actions", "frame_data"):
            value = getattr(frame, attr, None)
            parts.append(repr(value))
        return hashlib.sha1("|".join(parts).encode("utf-8")).hexdigest()

    @staticmethod
    def _legal_actions(latest_frame: FrameData) -> list[GameAction]:
        raw = getattr(latest_frame, "available_actions", None)
        if raw:
            legal = [a for a in raw if isinstance(a, GameAction)]
            if legal:
                return legal
        return [a for a in GameAction if a is not GameAction.RESET]

    def is_done(self, frames: list[FrameData], latest_frame: FrameData) -> bool:
        return latest_frame.state is GameState.WIN

    def choose_action(self, frames: list[FrameData], latest_frame: FrameData) -> GameAction:
        if latest_frame.state in (GameState.NOT_PLAYED, GameState.GAME_OVER):
            return GameAction.RESET

        signature = self._signature(latest_frame)
        if signature == self._last_signature:
            self._stagnant_steps += 1
        else:
            self._stagnant_steps = 0
        self._last_signature = signature
        self._frame_signatures.add(signature)

        legal = self._legal_actions(latest_frame)
        legal = [a for a in legal if a is not GameAction.RESET] or [GameAction.ACTION1]

        # Cover each currently legal simple action once. This is more
        # informative than random sampling and works across unknown games.
        unseen = [
            a for a in legal
            if not a.is_complex() and a.name not in self._seen_action_names
        ]
        if unseen:
            action = unseen[0]
        else:
            simple = [a for a in legal if not a.is_complex()]
            complex_actions = [a for a in legal if a.is_complex()]
            # Coordinate actions discover clickable targets, but do not
            # dominate the budget when no visual parser is available.
            if complex_actions and self._turn % self.PROBE_PERIOD == self.PROBE_PERIOD - 1:
                action = complex_actions[0]
            elif simple:
                action = self._rng.choice(simple)
            else:
                action = self._rng.choice(legal)

        self._turn += 1
        self._seen_action_names.add(action.name)
        if action.is_complex():
            # Deterministic board probes over the documented 64x64 range.
            k = self._turn
            x = (17 * k + 11 * (k // 4)) % 64
            y = (29 * k + 7 * (k // 3)) % 64
            action.set_data({"x": int(x), "y": int(y)})
            action.reasoning = {
                "policy": "systematic_coordinate_probe",
                "x": int(x),
                "y": int(y),
            }
        else:
            action.reasoning = {
                "policy": "legal_action_coverage",
                "turn": self._turn,
                "stagnant_steps": self._stagnant_steps,
            }
        return action


In [ ]:
import os

if os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    # Wait for the gateway sidecar to be ready.
    !curl --fail --retry 999 --retry-all-errors --retry-delay 5 \
          --retry-max-time 600 http://gateway:8001/api/games

    # Copy the framework into a writable location.
    !cp -r /kaggle/input/competitions/arc-prize-2026-arc-agi-3/ARC-AGI-3-Agents \
           /kaggle/working/ARC-AGI-3-Agents

    # Drop our agent in as a framework template.
    !cp /tmp/my_agent.py \
        /kaggle/working/ARC-AGI-3-Agents/agents/templates/my_agent.py

    # Register MyAgent in the framework's agent registry. We rewrite
    # __init__.py because the upstream version eagerly imports
    # templates with deps we don't ship (langgraph, smolagents, etc.).
    with open('/kaggle/working/ARC-AGI-3-Agents/agents/__init__.py', 'w') as f:
        f.write("""from typing import Type
from dotenv import load_dotenv
from .agent import Agent, Playback
from .swarm import Swarm
from .templates.random_agent import Random
from .templates.my_agent import MyAgent

load_dotenv()

AVAILABLE_AGENTS: dict[str, Type[Agent]] = {
    'random': Random,
    'myagent': MyAgent,
}
""")

    # Point the framework at the gateway sidecar.
    with open('/kaggle/working/ARC-AGI-3-Agents/.env', 'w') as f:
        f.write("""SCHEME=http
HOST=gateway
PORT=8001
ARC_API_KEY=test-key-123
ARC_BASE_URL=http://gateway:8001/
OPERATION_MODE=online
ENVIRONMENTS_DIR=
RECORDINGS_DIR=/kaggle/working/server_recording
""")

    # Run it. The gateway records every action and emits submission.parquet.
    !cd /kaggle/working/ARC-AGI-3-Agents && \
        MPLBACKEND=agg \
        python main.py --agent myagent


In [ ]:
import os
if not os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    # Save-and-run-all (commit) mode: emit a dummy submission so the
    # commit succeeds. The real submission.parquet is produced by the
    # gateway during competition rerun.
    import pandas as pd
    submission = pd.DataFrame(
        data=[['1_0', '1', True, 1]],
        columns=['row_id', 'game_id', 'end_of_game', 'score'])
    submission.to_parquet('/kaggle/working/submission.parquet', index=False)
    submission.head()
